<a href="https://colab.research.google.com/github/BF667/zero-shot-svc/blob/main/zero_shot_svc_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎤 Zero-Shot Singing Voice Conversion

Convert any singing voice to sound like a target speaker — **no training required**.

Based on RVC v2 architecture with RMVPE pitch extraction.

| Feature | Detail |
|---------|--------|
| Content Encoder | ContentVec (HuBERT) → 256-dim |
| F0 Extractor | RMVPE (Viterbi-smoothed) |
| Speaker Encoder | CAM++ → 192-dim |
| Generator | VITS + FiLM conditioning |
| Vocoder | HiFi-GAN (32kHz) |

---

In [ ]:
#@title ## 1. Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
#@title ## 2. Clone & Install

%cd /content
!git clone https://github.com/BF667/zero-shot-svc.git
%cd zero-shot-svc

!uv pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!uv pip install -q -r requirements.txt

In [ ]:
#@title ## 3. Download Pretrained Weights

#@markdown Downloads RMVPE, ContentVec, CAM++, and HiFi-GAN weights from HuggingFace (~2 GB, cached after first download).

from weights.download_weights import download_all
results = download_all(verbose=True)

In [ ]:
#@title ## 4. Upload Your Audio Files

#@markdown You need two audio files:
#@markdown - **Source audio**: The singing you want to convert (WAV recommended)
#@markdown - **Reference audio**: A short clip (5-15s) of the target voice

#@markdown Run the cell below to upload them via the file picker.
import os
from google.colab import files

os.makedirs('sample_audio', exist_ok=True)

print('Upload SOURCE audio (the singing to convert):')
uploaded_src = files.upload()
src_name = list(uploaded_src.keys())[0]
os.rename(src_name, f'sample_audio/{src_name}')
SOURCE_PATH = f'sample_audio/{src_name}'

print('\nUpload REFERENCE audio (target voice, 5-15s):')
uploaded_ref = files.upload()
ref_name = list(uploaded_ref.keys())[0]
os.rename(ref_name, f'sample_audio/{ref_name}')
REFERENCE_PATH = f'sample_audio/{ref_name}'

print(f'\n✓ Source:      {SOURCE_PATH}')
print(f'✓ Reference:   {REFERENCE_PATH}')

In [ ]:
#@title ## 5. init Voice Conversion

import sys
sys.path.insert(0, '/content/zero-shot-svc')

from pipeline.voice_converter import ZeroShotSVC

# Initialize on GPU
svc = ZeroShotSVC(device='cuda')
svc.load_models()

In [ ]:
#@title ##  Run Voice Conversion
output_path = svc.convert(
    source_path=SOURCE_PATH,
    reference_path=REFERENCE_PATH,
    output_path='sample_audio/converted.wav',
    f0_transpose=0,        # Change this: +12 = 1 octave up, -12 = 1 octave down
    noise_scale=0.4,       # 0.2 = conservative, 0.6 = more expressive
)

print(f'\n✓ Output saved to: {output_path}')

## 6. Listen & Download

In [ ]:
import IPython.display as ipd

print('Source (original singing):')
display(ipd.Audio(SOURCE_PATH))

print('Reference (target voice):')
display(ipd.Audio(REFERENCE_PATH))

print('Converted output:')
display(ipd.Audio(output_path))

In [ ]:
from google.colab import files
files.download(output_path)

In [ ]:
#@title ## 7. Advanced: Feature Analysis

#@markdown Inspect F0 pitch contours and content features.
import numpy as np
import matplotlib.pyplot as plt

# Extract features
features = svc.extract_features(SOURCE_PATH)
f0 = features['f0']
uv = features['uv']
voiced_f0 = f0[f0 > 0]

print(f'Duration:   {features["duration"]:.1f}s')
print(f'Content:    {features["content"].shape}')
print(f'F0 frames:  {len(f0)}')
if len(voiced_f0) > 0:
    print(f'F0 range:   {voiced_f0.min():.1f} – {voiced_f0.max():.1f} Hz')
    print(f'F0 mean:    {voiced_f0.mean():.1f} Hz')
    print(f'Voiced:     {len(voiced_f0)}/{len(f0)} ({100*len(voiced_f0)/len(f0):.0f}%)')

# Plot F0 contour
fig, ax = plt.subplots(figsize=(14, 3))
time_axis = np.arange(len(f0)) * 0.02  # 20ms per frame
voiced_mask = f0 > 0
ax.plot(time_axis[voiced_mask], f0[voiced_mask], 'b-', linewidth=0.8, label='F0')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('F0 Pitch Contour (RMVPE)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#@title ## 8. Advanced: Speaker Embedding

#@markdown Extract and compare speaker embeddings.

import numpy as np

emb = svc.extract_speaker_embedding(REFERENCE_PATH)
print(f'Embedding shape: {emb.shape}')
print(f'L2 norm:         {np.linalg.norm(emb):.4f}  (should be ~1.0)')
print(f'Range:           [{emb.min():.4f}, {emb.max():.4f}]')
print(f'Mean:            {emb.mean():.6f}')

# Cosine similarity between two embeddings
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

# Try with source audio too
emb_src = svc.extract_speaker_embedding(SOURCE_PATH)
sim = cosine_sim(emb, emb_src)
print(f'\nCosine similarity (source vs reference): {sim:.4f}')
print('  > 0.7 = same speaker,  < 0.5 = different speakers')

In [ ]:
#@title ## 9. Batch Conversion

#@markdown Convert multiple files at once using the same reference voice.


# Upload multiple source files
print('Upload multiple source audio files for batch conversion:')
batch_uploads = files.upload()

import os
os.makedirs('sample_audio/batch_output', exist_ok=True)

for fname in batch_uploads.keys():
    src = f'sample_audio/{fname}'
    os.rename(fname, src)
    out = f'sample_audio/batch_output/{fname}'
    print(f'\nConverting {fname}...')
    svc.convert(
        source_path=src,
        reference_path=REFERENCE_PATH,
        output_path=out,
    )
    print(f'  Done → {out}')

print(f'\n✓ Batch complete. {len(batch_uploads)} files converted.')

---

### Troubleshooting

| Problem | Solution |
|---------|----------|
| Output sounds like noise | Pretrained generator weights not loaded. This is expected with random init — load RVC community weights. |
| CUDA out of memory | Use smaller audio chunks or reduce batch size. Try `device='cpu'` as fallback. |
| Poor voice similarity | Use a longer/cleaner reference (10-15s, no background noise). |
| Octave errors in pitch | RMVPE handles most cases. For extreme registers, adjust `f0_min`/`f0_max` in config. |
| robotic/artificial sound | Lower `noise_scale` to 0.2-0.3 for more stable output. |

### References

- [RVC WebUI](https://github.com/RVC-Boss/Retrieval-based-Voice-Conversion-WebUI)
- [RMVPE Paper](https://arxiv.org/abs/2306.15412)
- [VITS Paper](https://arxiv.org/abs/2106.06103)
- [HiFi-GAN Paper](https://arxiv.org/abs/2010.05646)
- [CAM++ Paper](https://arxiv.org/abs/2210.16711)